In [47]:
# Imports
import os
import sys
import glob
import json
import sqlite3
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

In [48]:
# 1. Setup environment and download raw data
BASE_DIR = Path.cwd().resolve().parent
load_dotenv(dotenv_path=BASE_DIR / "config" / ".env")

# Azure Blob Storage connection
CONN_STR = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service = BlobServiceClient.from_connection_string(CONN_STR)

# Define raw data directories
RAW_TWITCH_DIR = BASE_DIR / "data" / "raw" / "twitch"
RAW_STEAM_DIR = BASE_DIR / "data" / "raw" / "steam"
RAW_CAFES_DIR = BASE_DIR / "data" / "raw" / "cafes"
# Ensure directories exist
RAW_TWITCH_DIR.mkdir(parents=True, exist_ok=True)
RAW_STEAM_DIR.mkdir(parents=True, exist_ok=True)
RAW_CAFES_DIR.mkdir(parents=True, exist_ok=True)

# Helper to download a container
def download_container(container_name: str, local_folder: Path):
    cc = blob_service.get_container_client(container_name)
    for blob in cc.list_blobs():
        dest = local_folder / blob.name
        with open(dest, "wb") as f:
            cc.download_blob(blob.name).readinto(f)
        print(f" {container_name}/{blob.name}")

print("Downloading from Azure Blob Storage...")
download_container("raw-twitch", RAW_TWITCH_DIR)
download_container("raw-steam", RAW_STEAM_DIR)
download_container("raw-cafes", RAW_CAFES_DIR)
print("\nAll data downloaded. Ready for Spark.")

 raw-twitch/game_ids_2026-05-01_015443.json
 raw-twitch/game_ids_2026-05-01_140409.json
 raw-twitch/game_ids_2026-05-01_140555.json
 raw-twitch/game_ids_2026-05-01_140827.json
 raw-twitch/game_ids_2026-05-01_141402.json
 raw-twitch/game_ids_2026-05-01_143748.json
 raw-twitch/game_ids_2026-05-01_144703.json
 raw-twitch/game_ids_2026-05-01_150115.json
 raw-twitch/game_ids_2026-05-01_151232.json
 raw-twitch/game_ids_2026-05-01_181607.json
 raw-twitch/streams_apex_legends_2026-05-01_015443.json
 raw-twitch/streams_apex_legends_2026-05-01_140409.json
 raw-twitch/streams_apex_legends_2026-05-01_140555.json
 raw-twitch/streams_apex_legends_2026-05-01_140827.json
 raw-twitch/streams_apex_legends_2026-05-01_141402.json
 raw-twitch/streams_apex_legends_2026-05-01_143748.json
 raw-twitch/streams_apex_legends_2026-05-01_144703.json
 raw-twitch/streams_apex_legends_2026-05-01_150115.json
 raw-twitch/streams_apex_legends_2026-05-01_151232.json
 raw-twitch/streams_apex_legends_2026-05-01_181607.json


In [49]:
# 2. Configure Spark session
venv_python = r"D:\\Msc_Data_Analytics\\Data_Intensive_Scalable_System\\CA2\\.venv_311\\Scripts\\python.exe"
os.environ["PYSPARK_PYTHON"] = venv_python
os.environ["PYSPARK_DRIVER_PYTHON"] = venv_python

spark = (SparkSession.builder
         .appName("DISS_CA2_SparkProcessing")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Version: {spark.version}")

Spark Version: 4.1.1


In [50]:
games_data = [
    (730,     "Counter-Strike",                  "Shooter",       "Valve",       True),
    (1172470, "Apex Legends",                    "Battle Royale", "Respawn",     True),
    (1172620, "Fortnite",                        "Battle Royale", "Epic Games",  True),
    (570,     "Dota 2",                          "MOBA",          "Valve",       True),
    (359550,  "Tom Clancy's Rainbow Six Siege",  "Shooter",       "Ubisoft",     True),
]
dim_game_schema = StructType([
    StructField("game_id",    IntegerType(), False),
    StructField("title",      StringType(),  False),
    StructField("genre",      StringType(),  True),
    StructField("developer",  StringType(),  True),
    StructField("is_esports", BooleanType(), True),
])
dim_game = spark.createDataFrame(games_data, dim_game_schema)
print("dim_game")
dim_game.show(truncate=False)

dim_game
+-------+------------------------------+-------------+----------+----------+
|game_id|title                         |genre        |developer |is_esports|
+-------+------------------------------+-------------+----------+----------+
|730    |Counter-Strike                |Shooter      |Valve     |true      |
|1172470|Apex Legends                  |Battle Royale|Respawn   |true      |
|1172620|Fortnite                      |Battle Royale|Epic Games|true      |
|570    |Dota 2                        |MOBA         |Valve     |true      |
|359550 |Tom Clancy's Rainbow Six Siege|Shooter      |Ubisoft   |true      |
+-------+------------------------------+-------------+----------+----------+



In [51]:
cafes_path = RAW_CAFES_DIR / "cafes_ireland.json"
print(f"Reading: {cafes_path.name}")

dim_cafe = (spark.read
            .option("multiline", "true")
            .json(str(cafes_path)))

dim_cafe = (dim_cafe
    .withColumn("cafe_id",   F.col("cafe_id").cast(IntegerType()))
    .withColumn("capacity",  F.col("capacity").cast(IntegerType()))
    .withColumn("latitude",  F.col("latitude").cast(DoubleType()))
    .withColumn("longitude", F.col("longitude").cast(DoubleType()))
    .select("cafe_id","name","city","country","latitude","longitude","capacity","features")
)
print("dim_cafe")
dim_cafe.show(truncate=False)

Reading: cafes_ireland.json
dim_cafe
+-------+-------------------+---------+-------+--------+---------+--------+----------------+
|cafe_id|name               |city     |country|latitude|longitude|capacity|features        |
+-------+-------------------+---------+-------+--------+---------+--------+----------------+
|1      |Pixel Palace       |Dublin   |Ireland|53.3498 |-6.2603  |40      |Streaming, LAN  |
|2      |LevelUp Lounge     |Dublin   |Ireland|53.3401 |-6.2611  |30      |LAN, Snack Bar  |
|3      |GameZone Cork      |Cork     |Ireland|51.8985 |-8.4756  |25      |Streaming       |
|4      |EsportsHub Galway  |Galway   |Ireland|53.2743 |-9.0514  |20      |Tournaments     |
|5      |Arena Limerick     |Limerick |Ireland|52.668  |-8.6305  |35      |LAN, Streaming  |
|6      |Nexus Gaming       |Belfast  |Ireland|54.5973 |-5.9301  |50      |VR, Streaming   |
|7      |RetroPlay Waterford|Waterford|Ireland|52.2593 |-7.1101  |15      |LAN             |
|8      |ProGamer Kilkenny  |Kilk

In [52]:
twitch_files = sorted(glob.glob(str(RAW_TWITCH_DIR / "twitch_streams_flat_*.csv")))
assert twitch_files, "No Twitch flat CSV found!"
latest = twitch_files[-1]
print(f"Reading: {Path(latest).name}")


spark.conf.set("spark.sql.ansi.enabled", False)

twitch_raw = (spark.read
              .option("header", "true")
              .option("inferSchema", "false")  
              .csv(latest))

print(f"Raw rows: {twitch_raw.count()}")

# Cast viewer_count explicitly to integer, bad values become NULL
twitch_raw = twitch_raw.withColumn(
    "viewer_count", F.col("viewer_count").cast(IntegerType())
)

# Join on game_name - game_id
twitch_mapped = twitch_raw.join(
    dim_game.select(F.col("game_id"), F.col("title").alias("game_name")),
    on="game_name", how="inner"
)

snapshot_date = twitch_raw.select("snapshot_time_utc").first()[0][:10]

fact_streaming = (twitch_mapped
    .groupBy("game_id")
    .agg(
        F.avg("viewer_count").cast(IntegerType()).alias("avg_viewers"),
        F.max("viewer_count").cast(IntegerType()).alias("peak_viewers"),
        F.count("stream_id").cast(IntegerType()).alias("stream_count"),
        F.first("language").alias("top_region"),
    )
    .withColumn("date", F.lit(snapshot_date))
    .withColumn("hours_watched",
        (F.col("avg_viewers") * F.col("stream_count") / 60).cast(IntegerType()))
    .select("game_id","date","avg_viewers","peak_viewers","hours_watched","stream_count","top_region")
)

print("fact_streaming_metrics")
fact_streaming.show(truncate=False)


Reading: twitch_streams_flat_2026-05-01_181607.csv
Raw rows: 491
fact_streaming_metrics
+-------+----------+-----------+------------+-------------+------------+----------+
|game_id|date      |avg_viewers|peak_viewers|hours_watched|stream_count|top_region|
+-------+----------+-----------+------------+-------------+------------+----------+
|570    |2026-05-01|357        |4787        |583          |98          |ru        |
|730    |2026-05-01|704        |11983       |1126         |96          |ru        |
|359550 |2026-05-01|108        |3484        |180          |100         |en        |
|1172470|2026-05-01|156        |2661        |260          |100         |en        |
|1172620|2026-05-01|740        |12462       |1196         |97          |pl        |
+-------+----------+-----------+------------+-------------+------------+----------+



In [53]:
sentiment_path = RAW_STEAM_DIR / "steam_sentiment_summary.csv"
assert sentiment_path.exists(), "steam_sentiment_summary.csv not found!"
print(f"Reading: {sentiment_path.name}")

steam_raw = (spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(str(sentiment_path)))

steam_raw.printSchema()

fact_reviews = (steam_raw
    .withColumnRenamed("appid", "game_id")
    .withColumn("game_id",         F.col("game_id").cast(IntegerType()))
    .withColumn("review_count",    F.col("review_count").cast(IntegerType()))
    .withColumn("sentiment_score", F.col("sentiment_score").cast(DoubleType()))
    .withColumn("review_velocity", F.col("review_velocity").cast(DoubleType()))
    .withColumn("date",            F.lit(snapshot_date))
    .select("game_id","date","review_count","sentiment_score","review_velocity")
)

print("fact_review_engagement")
fact_reviews.show(truncate=False)

Reading: steam_sentiment_summary.csv
root
 |-- appid: integer (nullable = true)
 |-- game_name: string (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- positive_count: integer (nullable = true)
 |-- avg_playtime: double (nullable = true)
 |-- sentiment_score: double (nullable = true)
 |-- review_velocity: double (nullable = true)
 |-- avg_playtime_hrs: double (nullable = true)

fact_review_engagement
+-------+----------+------------+---------------+---------------+
|game_id|date      |review_count|sentiment_score|review_velocity|
+-------+----------+------------+---------------+---------------+
|570    |2026-05-01|200         |0.7            |28.6           |
|730    |2026-05-01|200         |0.725          |28.6           |
|359550 |2026-05-01|200         |0.685          |28.6           |
|1172470|2026-05-01|200         |0.595          |28.6           |
|1172620|2026-05-01|200         |0.66           |28.6           |
+-------+----------+------------+---------------+

In [54]:
hype_path = RAW_STEAM_DIR / "games_stats_hype.json"
assert hype_path.exists(), "games_stats_hype.json not found!"
print(f"Reading: {hype_path.name}")

hype_raw = (spark.read
            .option("multiline", "true")
            .json(str(hype_path)))

hype_raw.printSchema()

fact_hype = (hype_raw
    .join(dim_game.select(F.col("game_id"), F.col("title").alias("game_name")),
          on="game_name", how="inner")
    .withColumn("followers",       F.col("followers").cast(IntegerType()))
    .withColumn("follower_growth", F.col("follower_growth").cast(IntegerType()))
    .withColumn("pre_release_flag",F.col("pre_release_flag").cast(BooleanType()))
    .select("game_id","followers","follower_growth","pre_release_flag")
)

print("fact_hype_metrics")
fact_hype.show(truncate=False)

Reading: games_stats_hype.json
root
 |-- appid: long (nullable = true)
 |-- follower_growth: long (nullable = true)
 |-- followers: long (nullable = true)
 |-- game_name: string (nullable = true)
 |-- pre_release_flag: boolean (nullable = true)

fact_hype_metrics
+-------+---------+---------------+----------------+
|game_id|followers|follower_growth|pre_release_flag|
+-------+---------+---------------+----------------+
|730    |1900000  |500            |false           |
|1172470|1200000  |350            |false           |
|1172620|2100000  |1100           |true            |
|570    |3200000  |700            |false           |
|359550 |1500000  |420            |false           |
+-------+---------+---------------+----------------+



In [55]:

print("ANALYSIS 1 - Streaming Demand vs Cafe Coverage")

game_demand = (fact_streaming
    .groupBy("game_id")
    .agg(
        F.avg("avg_viewers").alias("avg_viewers"),
        F.max("peak_viewers").alias("peak_viewers"),
        F.sum("stream_count").alias("total_streams")
    )
    .join(dim_game.select("game_id","title"), on="game_id")
    .orderBy(F.desc("avg_viewers"))
)
game_demand.show(truncate=False)

cafe_coverage = (dim_cafe
    .groupBy("city")
    .agg(
        F.count("cafe_id").alias("cafe_count"),
        F.sum("capacity").alias("total_capacity")
    )
    .orderBy(F.desc("total_capacity"))
)
print("Cafe Coverage per City:")
cafe_coverage.show(truncate=False)

ANALYSIS 1 - Streaming Demand vs Cafe Coverage
+-------+-----------+------------+-------------+------------------------------+
|game_id|avg_viewers|peak_viewers|total_streams|title                         |
+-------+-----------+------------+-------------+------------------------------+
|1172620|740.0      |12462       |97           |Fortnite                      |
|730    |704.0      |11983       |96           |Counter-Strike                |
|570    |357.0      |4787        |98           |Dota 2                        |
|1172470|156.0      |2661        |100          |Apex Legends                  |
|359550 |108.0      |3484        |100          |Tom Clancy's Rainbow Six Siege|
+-------+-----------+------------+-------------+------------------------------+

Cafe Coverage per City:
+---------+----------+--------------+
|city     |cafe_count|total_capacity|
+---------+----------+--------------+
|Dublin   |2         |70            |
|Belfast  |1         |50            |
|Limerick |1      

In [56]:
print("ANALYSIS 2 - Event Impact / Viewer Growth Projection")

event_impact = (fact_streaming
    .join(fact_hype.select("game_id","follower_growth"), on="game_id")
    .withColumn("viewers_pre",  F.col("avg_viewers"))
    .withColumn("viewers_post",
        (F.col("avg_viewers") * (1 + F.col("follower_growth") / 100000))
        .cast(IntegerType()))
    .withColumn("growth_pct",
        F.round(
            (F.col("viewers_post") - F.col("viewers_pre")) /
             F.col("viewers_pre") * 100, 2))
    .join(dim_game.select("game_id","title"), on="game_id")
    .select("game_id","title","viewers_pre","viewers_post","growth_pct")
    .orderBy(F.desc("growth_pct"))
)
event_impact.show(truncate=False)

ANALYSIS 2 - Event Impact / Viewer Growth Projection
+-------+------------------------------+-----------+------------+----------+
|game_id|title                         |viewers_pre|viewers_post|growth_pct|
+-------+------------------------------+-----------+------------+----------+
|1172620|Fortnite                      |740        |748         |1.08      |
|570    |Dota 2                        |357        |359         |0.56      |
|730    |Counter-Strike                |704        |707         |0.43      |
|1172470|Apex Legends                  |156        |156         |0.0       |
|359550 |Tom Clancy's Rainbow Six Siege|108        |108         |0.0       |
+-------+------------------------------+-----------+------------+----------+



In [57]:
print("ANALYSIS 3 - Hybrid Opportunity Score per Game")

# game_demand already has 'title' from Analysis 1 join - drop it before re-joining
game_demand_no_title = game_demand.drop("title")

combined = (game_demand_no_title
    .join(fact_reviews.select("game_id","sentiment_score","review_velocity"), on="game_id")
    .join(fact_hype.select("game_id","followers","follower_growth"), on="game_id")
)

max_viewers   = combined.agg(F.max("avg_viewers")).collect()[0][0]
max_sentiment = combined.agg(F.max("sentiment_score")).collect()[0][0]
max_followers = combined.agg(F.max("followers")).collect()[0][0]

opportunity_scores = (combined
    .withColumn("streaming_index",  F.col("avg_viewers")     / max_viewers)
    .withColumn("engagement_index", F.col("sentiment_score") / max_sentiment)
    .withColumn("hype_index",       F.col("followers")        / max_followers)
    .withColumn("opportunity_score",
        F.round(
            F.col("streaming_index")  * 0.5 +
            F.col("engagement_index") * 0.3 +
            F.col("hype_index")       * 0.2, 4))
    .join(dim_game.select("game_id","title"), on="game_id")
    .select("game_id","title","streaming_index","engagement_index","hype_index","opportunity_score")
    .orderBy(F.desc("opportunity_score"))
)
opportunity_scores.show(truncate=False)

ANALYSIS 3 - Hybrid Opportunity Score per Game
+-------+------------------------------+-------------------+------------------+----------+-----------------+
|game_id|title                         |streaming_index    |engagement_index  |hype_index|opportunity_score|
+-------+------------------------------+-------------------+------------------+----------+-----------------+
|1172620|Fortnite                      |1.0                |0.9103448275862069|0.65625   |0.9044           |
|730    |Counter-Strike                |0.9513513513513514 |1.0               |0.59375   |0.8944           |
|570    |Dota 2                        |0.48243243243243245|0.9655172413793103|1.0       |0.7309           |
|359550 |Tom Clancy's Rainbow Six Siege|0.14594594594594595|0.9448275862068967|0.46875   |0.4502           |
|1172470|Apex Legends                  |0.21081081081081082|0.8206896551724138|0.375     |0.4266           |
+-------+------------------------------+-------------------+------------------+--

In [58]:
DB_PATH = BASE_DIR / "data" / "hybrid_esports.db"
conn = sqlite3.connect(str(DB_PATH))

dim_game          .toPandas().to_sql("dim_game",                conn, if_exists="replace", index=False)
dim_cafe          .toPandas().to_sql("dim_cafe",                conn, if_exists="replace", index=False)
fact_streaming    .toPandas().to_sql("fact_streaming_metrics",  conn, if_exists="replace", index=False)
fact_reviews      .toPandas().to_sql("fact_review_engagement",  conn, if_exists="replace", index=False)
fact_hype         .toPandas().to_sql("fact_hype_metrics",       conn, if_exists="replace", index=False)
opportunity_scores.toPandas().to_sql("fact_opportunity_scores", conn, if_exists="replace", index=False)

conn.commit()
conn.close()
print(f"All 6 tables written to SQLite: {DB_PATH.name}")
spark.stop()

All 6 tables written to SQLite: hybrid_esports.db
